# Biomedical Data Bases, 2025-2026
###  Create Your Own Database
These are notes by prof. Davide Salomoni (d.salomoni@unibo.it) for the Biomedical Data Base course at the University of Bologna, academic year 2025-2026.

In [ ]:
# import the necessary modules
import sqlite3 as sql
import requests

### Create the SQLite schema

We will create the schema for an SQLite DB called `my_database.sqlite`. Refer to the slides for details about the schema.

In [ ]:
# create the schema
conn = sql.connect('my_database.sqlite')
cur = conn.cursor()

cur.execute('''DROP TABLE IF EXISTS structures''')
cur.execute('''DROP TABLE IF EXISTS polymers''')
cur.execute('''DROP TABLE IF EXISTS go_annotations''')

cur.execute('''CREATE TABLE structures (
    pdb_id TEXT PRIMARY KEY,
    title TEXT,
    total_weight REAL,
    atom_count INTEGER,
    residue_count INTEGER
    )''')
cur.execute('''
CREATE TABLE polymers (
    polymer_id TEXT PRIMARY KEY,
    pdb_id TEXT NOT NULL,
    uniprot_accession TEXT,
    protein_name TEXT,
    scientific_name TEXT,
    FOREIGN KEY (pdb_id) REFERENCES structures(pdb_id),
    UNIQUE (polymer_id, scientific_name, uniprot_accession)
    )''')
cur.execute('''CREATE TABLE go_annotations (
    id INTEGER PRIMARY KEY,
    go_id TEXT NOT NULL,  
    go_term TEXT NOT NULL,
    go_source TEXT NOT NULL,
    polymer_id TEXT NOT NULL,
    FOREIGN KEY (polymer_id) REFERENCES polymers(polymer_id),
    UNIQUE (polymer_id, go_id)
    )''')
# commit the schema to the DB
conn.commit()

### Query PDB, Uniprot and store the results in SQLite

In [ ]:
pdb_query = '''
{
  entries(entry_ids: ["4GYD", "1TU2"]) {
    rcsb_id
    struct { title }
    rcsb_entry_info {
      molecular_weight
      deposited_atom_count
      deposited_modeled_polymer_monomer_count
    }
    polymer_entities {
      rcsb_id
      rcsb_entity_source_organism {
        ncbi_scientific_name
      }
      uniprots {
        rcsb_uniprot_container_identifiers {
          uniprot_id
        }
        rcsb_uniprot_protein {
          name {
            value
          }
        }
      }
    }
  }
}
'''
# get the PDB data with GraphQL
p = requests.get('https://data.rcsb.org/graphql?query=%s' % requests.utils.requote_uri(pdb_query))
j = p.json()

In [ ]:
# which keys are there?
j.keys()

In [ ]:
# explore what the returned data looks like:
# it is a set of nested Python data structures;
# we need to extract the values we need
j['data']

In [ ]:
# for example, extract some macromolecule parameters
for prot in (j['data']['entries']):
    # each entry corresponds to a single PDB ID
    print("%s (%s): " % (prot['rcsb_id'], prot['struct']['title']))
    print("Macromolecule parameters:")
    print("  molecular weight (kDa): %s" % prot['rcsb_entry_info']['molecular_weight'])
    print("  deposited atom count: %s\n" % prot['rcsb_entry_info']['deposited_atom_count'])

In [ ]:
# extract data and update the SQLite database
# the print() statements below are for explanatory purposes

for prot in j['data']['entries']:
    # structures
    pdb_id = prot['rcsb_id']
    title = prot['struct']['title']
    print("PDB: %s (%s)" % (pdb_id, title))
    weight = prot['rcsb_entry_info']['molecular_weight']
    atom_count = prot['rcsb_entry_info']['deposited_atom_count']
    residue_count = prot['rcsb_entry_info']['deposited_modeled_polymer_monomer_count']
    # store into the table structures
    cur.execute('''INSERT INTO structures VALUES (?, ?, ?, ?, ?)''',
                (pdb_id,
                title,
                weight,
                atom_count,
                residue_count)
               )
    # polymers
    for polymer in prot['polymer_entities']:
        polymer_id = polymer['rcsb_id']
        # extract all source organisms
        source_organisms = list()
        for so in polymer['rcsb_entity_source_organism']:
            # scientific name(s)
            source_organisms.append(so['ncbi_scientific_name'])
        # extract all uniprots
        uniprots = list()
        for up in polymer['uniprots']:
            # uniprot accession id and protein name
            uniprots.append((up['rcsb_uniprot_container_identifiers']['uniprot_id'],
                            up['rcsb_uniprot_protein']['name']['value']))
        combinations = [(a, b) for a in source_organisms for b in uniprots]
        # store into the table polymers
        for (s,u) in combinations:
            cur.execute('''INSERT INTO polymers VALUES (?, ?, ?, ?, ?)''',
                        (polymer_id,
                        pdb_id,
                        u[0], # uniprot accession ID
                        u[1], # protein name
                        s)    # source organism name
                       )
        # go annotations
        for up in uniprots:
            accession_id = up[0]  # uniprot accession ID
            uniprot_url = 'https://www.ebi.ac.uk/proteins/api/proteins?offset=0&size=10&accession=%s' % accession_id
            r = requests.get(uniprot_url, headers={"Accept" : "application/json"})
            # the Gene Ontology information is stored in the 'dbReferences' structure (see slides)
            db_info = r.json()[0]['dbReferences']
            for db in db_info:
                if db['type'] == 'GO':
                    # it is a Gene Ontology entry
                    go_id = db['id']
                    go_term = db['properties']['term']
                    go_source = db['properties']['source']
                    print(polymer_id, go_id, go_term, go_source)
                    # store into the table go_annotations
                    # the column "id" is the auto-incremented primary key, so we don't specify it
                    cur.execute('''INSERT INTO go_annotations (go_id, go_term, go_source, polymer_id) 
                                   VALUES (?, ?, ?, ?)''',
                                (go_id,        # GO annotation id
                                 go_term,      # GO term
                                 go_source,    # GO source
                                 polymer_id)   # polymer id
                               )
conn.commit()

### Performing queries on the SQLlite database

In [ ]:
# all stored attributes of a given PDB ID
cur.execute(''' SELECT * FROM structures WHERE pdb_id = ? ''', ("4GYD",))
print(cur.fetchall())

In [ ]:
# all polymers for a given PDB ID
cur.execute(''' SELECT * FROM polymers WHERE pdb_id='4GYD' ''')
print(cur.fetchall())

In [ ]:
# all GO entries for a given Uniprot ID:
cur.execute(''' SELECT go_id FROM go_annotations AS ga
                WHERE ga.polymer_id IN (
                    SELECT p.polymer_id
                    FROM polymers AS p
                    WHERE p.uniprot_accession = ?
                )''',
            ("P46444",))
print(cur.fetchall())

In [ ]:
# the top 10 heaviest structures
cur.execute(''' SELECT pdb_id, title, total_weight FROM structures
                ORDER BY total_weight DESC
                LIMIT 10
                ''')
print(cur.fetchall())

In [ ]:
# All GO annotations coming from a given source
cur.execute(''' SELECT * from go_annotations WHERE go_source LIKE "%UniprotKB-UniRule%" ''')
print(cur.fetchall())

In [ ]:
# Count GO annotations per structure

cur.execute("""
    SELECT COUNT(*)
    FROM go_annotations
    WHERE polymer_id IN (
        SELECT polymer_id
        FROM polymers
        WHERE pdb_id = ?
    )
""", ("1TU2",))
print(cur.fetchall())

In [ ]:
# all GO entries for a given Uniprot ID, using JOIN
cur.execute(''' SELECT g.go_id
                FROM go_annotations AS g
                JOIN polymers AS p ON p.polymer_id = g.polymer_id
                WHERE p.uniprot_accession = ? ''', ("P46444",))
print(cur.fetchall())

In [ ]:
# remember to close the connection at the end
conn.close()